In [1]:
import pandas as pd

In [6]:
df = pd.read_csv ('../quantification/results/classification_results_2026-01-23_19-54.csv')

In [7]:
df['Dataset'] = df['Dataset'].str.replace('_', ' ').str.title()
df['Split'] = df['Split'].str.replace('_', ' ').str.title()

In [9]:
pivot = df.pivot_table(index=['Dataset', 'Split'],
                       columns='Model',
                       values='Macro F1',
                       aggfunc='first')
pivot = pivot[['MLP', 'GCN', 'SAGE']]

In [10]:
def bold_max(row):
    m = row.max()
    return [f"\\textbf{{{x:.4f}}}" if x == m else f"{x:.4f}" for x in row]

formatted_data = pivot.apply(bold_max, axis=1, result_type='expand')
formatted_data.columns = [f"\\multicolumn{{1}}{{c}}{{{col}}}" for col in pivot.columns]

In [11]:
latex_code = formatted_data.to_latex(
    multirow=True,
    index=True,
    float_format="%.4f",
    column_format='ll|rrr',
    escape=False
)
latex_code = latex_code.replace(r'\toprule', '', 1)
latex_code = latex_code.replace(r'\midrule', r'\hline')
latex_code = latex_code.replace(r'\bottomrule', r'\hline')

print(latex_code)

\begin{tabular}{ll|rrr}

 &  & \multicolumn{1}{c}{MLP} & \multicolumn{1}{c}{GCN} & \multicolumn{1}{c}{SAGE} \\
Dataset & Split &  &  &  \\
\hline
\multirow[t]{6}{*}{Deezer Europe} & Split 0 & \textbf{0.5713} & 0.4678 & 0.5204 \\
 & Split 1 & \textbf{0.5695} & 0.4852 & 0.5651 \\
 & Split 2 & \textbf{0.6140} & 0.5487 & 0.6016 \\
 & Split 3 & 0.3471 & \textbf{0.5069} & 0.4619 \\
 & Split 4 & \textbf{0.5569} & 0.5051 & 0.5291 \\
 & Split 5 & \textbf{0.6165} & 0.5516 & 0.6062 \\
\cline{1-5}
\multirow[t]{3}{*}{Ogbn Arxiv} & Split 0 & 0.3040 & 0.3001 & \textbf{0.3125} \\
 & Split 1 & 0.3201 & 0.3781 & \textbf{0.3916} \\
 & Split 2 & 0.3121 & 0.3186 & \textbf{0.3459} \\
\cline{1-5}
\multirow[t]{4}{*}{Presidential Election} & Split 0 & 0.6099 & 0.6282 & \textbf{0.7763} \\
 & Split 1 & 0.8046 & 0.7159 & \textbf{0.8454} \\
 & Split 2 & 0.8161 & 0.7060 & \textbf{0.8261} \\
 & Split 3 & 0.7583 & 0.7419 & \textbf{0.8703} \\
\cline{1-5}
\multirow[t]{5}{*}{Twitch Gamers} & Split 0 & \textbf{0.5652} & 

In [73]:
#pivot.to_csv('../quantification/results/pivoted_results.csv')

In [143]:
df = pd.read_csv('../quantification/results/quantification_results.csv')

In [1]:
def generate_dataset_table(dataframe, dataset_name):
    ds_df = dataframe[dataframe['Dataset'] == dataset_name].copy()
    pivot_df = ds_df.pivot_table(
        index=['Split', 'Classifier'],
        columns=['Method'],
        values=['MAE', 'KL']
    )
    pivot_df.columns = pivot_df.columns.swaplevel(0, 1)
    pivot_df.sort_index(axis=1, inplace=True)
    pivot_df = pivot_df.reindex(columns=['MAE', 'KL'], level=1)
    methods = pivot_df.columns.levels[0]
    num_methods = len(methods)
    column_def = "ll" + "|cc" * num_methods + "|"
    header_cells = [r"\multicolumn{2}{c|}{\textbf{" + m + "}}" for m in methods]
    header_row_1 = " &  & " + " & ".join(header_cells) + r" \\"
    metrics_cells = ["MAE & KL"] * num_methods
    header_row_2 = r"\textbf{Split} & \textbf{Classifier} & " + " & ".join(metrics_cells) + r" \\"

    body = pivot_df.to_latex(
        float_format="%.4f",
        header=False,
        index=True,
        index_names=False,
        multirow=True,
        column_format=None
    )

    lines = body.splitlines()
    clean_body_lines = [l for l in lines if "tabular" not in l and "toprule" not in l and "bottomrule" not in l]
    clean_body = "\n".join(clean_body_lines)

    # 5. Assemble Final LaTeX Block
    latex_code = f"""
% --- TABLE FOR {dataset_name.upper()} ---
\\begin{{table}}[h!]
\\centering
\\scriptsize % Smaller font to fit width
\\setlength{{\\tabcolsep}}{{3pt}} % Adjust padding
\\renewcommand{{\\arraystretch}}{{1.1}} % Breathing room for rows

\\resizebox{{\\textwidth}}{{!}}{{%
    \\begin{{tabular}}{{{column_def}}}
    \\toprule
    {header_row_1}
    \\midrule
    {header_row_2}
    \\midrule
{clean_body}
    \\bottomrule
    \\end{{tabular}}%
}}
\\caption{{Quantification results for \\textbf{{{dataset_name}}} (Methods separated by vertical lines).}}
\\end{{table}}
"""
    return latex_code

# ---------------------------------------------------------
# EXECUTE FOR EACH DATASET
# ---------------------------------------------------------

unique_datasets = df['Dataset'].unique()

for ds in unique_datasets:
    print("\n\n")

NameError: name 'df' is not defined